# ML-Driven NoSQL Pattern Selection for Retrieval-Augmented Generation Workloads

**Google Colab implementation notebook — research prototype and experiment pipeline**

> **Research-integrity status:** This notebook is designed to collect and analyse genuine measurements from configured database services. It does not contain pre-filled benchmark results. Until database endpoints are configured and experiments are executed, result tables and trained-model sections will remain empty or will stop with a clear message.

**Research objective:** Learn from measured workload/database performance and build an explainable recommender that selects a suitable NoSQL database pattern for a given workload. Validate vector retrieval separately through an actual BEIR/Qdrant retrieval experiment.

**Candidate systems:** Redis (key-value), MongoDB (document), Apache Cassandra (wide-column), Neo4j (graph), and Qdrant (vector).

**Notebook mode:** staged implementation, pilot-first, resumable experiment batches.

## 0. Read this before running

### What this notebook does

1. Checks the Colab runtime and prepares persistent output folders.
2. Installs Python drivers and analytical libraries.
3. Reads database connection settings from Colab Secrets or environment variables.
4. Connects to Redis, MongoDB, Cassandra, Neo4j and Qdrant when endpoints are supplied.
5. Runs a controlled pilot for a common logical record workload across Redis, MongoDB, Cassandra and Neo4j.
6. Runs a separate native approximate-nearest-neighbour (ANN) benchmark against Qdrant.
7. Builds a workload-to-best-store dataset from successful measured runs.
8. Compares Decision Tree, Random Forest and Gradient Boosting classifiers.
9. Computes held-out latency regret against simple baselines.
10. Produces SHAP explanations for the selected tree-based model.
11. Optionally downloads the BEIR SciFact dataset and evaluates Qdrant retrieval quality.
12. Exports raw data, processed data, models, plots, logs and a reproducibility manifest.

### What it cannot do without external services

Colab is not a managed Docker host for this experiment. The notebook does not try to launch five database containers inside the Colab runtime. You must provide reachable database endpoints, preferably hosted in the same controlled environment/region. Use TLS and authentication. Do not expose database ports publicly without a secure access plan.

Google's Colab FAQ states that compute resources are dynamic and not guaranteed, and that managed runtimes have usage and execution limits. A GPU is not required for the tabular ML part of this project. See: https://research.google.com/colaboratory/faq.html

### Important measurement caveats

- This notebook's common CRUD benchmark compares the same logical record read/update task mapped to each database's native representation. It is a controlled application-level comparison, not a universal claim that one engine is best for every database workload.
- Qdrant is tested separately for vector ANN retrieval. Do not compare its ANN latency directly with key-value CRUD latency.
- Latency measured from Colab to remote databases includes network path and service overhead. Keep endpoints co-located where possible, use the same client region where practical, record configuration/version information, and interpret cross-provider comparisons cautiously.
- Client CPU and memory are measured inside Colab. They are not database-server resource measurements. Server-side resource metrics require monitoring on the database host.
- The default pilot is deliberately small. A 300+ configuration target must be earned through distinct, recorded workload/database configurations, not generated as fake rows. Repeated trials are repetitions, not new configurations.
- Consistency and vector-dimension features are recorded in the schema, but the default CRUD pilot does not actively vary database consistency policies or vector indexes. Do not use such unvaried fields as ML predictors. The vector experiment is separate.

### Suggested execution order

Run sections 1–8 first. Configure secrets and confirm database connectivity. Run the pilot. Inspect failures and data quality. Only then enable the full benchmark matrix. Train ML models only after a sufficient measured dataset exists.

## 1. Research question, scope and method

### Research question

Can workload characteristics predict which feasible database pattern performs best, and can the resulting recommendation be explained to a developer?

### Research gaps from the supplied research brief

- Database selection guidance is frequently qualitative or vendor-driven.
- Vector stores are often studied separately from key-value, document and graph patterns.
- Database recommenders need better explainability and regret-based evaluation.
- End-to-end validation in RAG pipelines is limited.

### Proposed experimental stages

**Stage A — common logical record workload:** Redis, MongoDB, Cassandra and Neo4j are benchmarked on a controlled key/value-like record read/update task. MongoDB stores structured documents; the other engines use their native representation. Workload features include read ratio, write ratio, query family, record payload size, number of records and client concurrency.

**Stage B — vector retrieval:** Qdrant is benchmarked on vector upsert/search with exact-neighbour ground truth. This produces vector-specific measurements such as Recall@K and query latency.

**Stage C — RAG retrieval validation:** BEIR SciFact corpus and queries are embedded with a fixed sentence-transformer model and indexed in Qdrant. Recall@K, nDCG@K, MRR and query latency are calculated.

### Important scope distinction

The ML classifier in the default version is trained only on workloads where multiple candidate databases have been measured on the same logical task. Qdrant's separate ANN measurements are not mixed with CRUD latency labels. To train a unified vector-store selector in a later iteration, additional vector-capable backends (for example, Redis Stack vector search, Neo4j vector indexes, or MongoDB Atlas Vector Search) must be implemented and benchmarked under the same retrieval protocol. Do not claim a five-engine, all-workload classifier until this is done.

## 2. Runtime setup and experiment controls

The notebook supports a **pilot-first** workflow. Change the flags below deliberately. Do not turn on the full matrix until the pilot is stable and the database configuration has been documented.

The database services must already exist outside the Colab managed runtime. The notebook will connect to them using secrets configured in the Colab key panel.

In [36]:
from pathlib import Path
import os, sys, json, time, platform, subprocess, textwrap, warnings, hashlib, random, re, uuid
from datetime import datetime, timezone

# ==============================================================================
# Core execution flags. All expensive operations are disabled by default.
# ==============================================================================
RUN_SMOKE_TEST = True            # Safe, DB-independent verification of package, CLI, and Drive persistence
RUN_CONNECTION_CHECKS = True     # Non-destructive ping/health-checks of configured endpoints
RUN_PILOT = False                # Pre-flight 8-run pilot on candidate databases (250 ops/run)
RUN_FULL_MATRIX = False          # Full 75-profile x 4-engine x 3-repetition benchmark matrix (900 runs)
RUN_QDRANT_ANN_PILOT = False     # Native vector ANN latency & Recall@K benchmark on Qdrant
RUN_BEIR_RAG = False             # End-to-end passage retrieval validation on BEIR SciFact
RUN_ML_TRAINING = False          # Train Decision Tree, Random Forest, Gradient Boosting on measured labels
RUN_SHAP = False                 # Multi-class TreeExplainer SHAP explainability analysis

# Pilot defaults (small by design)
PILOT_WORKLOAD_PROFILES = 2
PILOT_OPERATIONS = 250
PILOT_RECORDS = 200
PILOT_REPETITIONS = 1
PILOT_CONCURRENCY = 1

# Full benchmark target (75 profiles x 4 engines = 300 configurations x 3 repetitions = 900 runs)
FULL_WORKLOAD_PROFILES = 75
FULL_OPERATIONS_PER_RUN = 1500
FULL_RECORDS_PER_RUN = 1000
FULL_REPETITIONS = 3
FULL_BATCH_SIZE = 5              # Save after every small batch to prevent loss on runtime reset
FULL_CONCURRENCY_LEVELS = [1, 4]
CLEANUP_AFTER_RUN = True         # Clean temporary tables/keys after measurement; raw results persist

# Quality and objective policy
PRIMARY_OBJECTIVE = "min_p95_latency_ms"
MAX_ALLOWED_ERROR_RATE = 0.01
MIN_REQUIRED_RECALL_AT_K = 0.90
RANDOM_SEED = 42

print("Python:", sys.version)
print("Platform:", platform.platform())
print("UTC start:", datetime.now(timezone.utc).isoformat())

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Platform: Linux-6.6.159+-x86_64-with-glibc2.39
UTC start: 2026-11-13T07:23:16.762281+00:00


## 3. Check Colab hardware and prepare persistent storage

Google Colab runtimes can be reset. Save outputs frequently. Google Drive is optional but recommended for long experiments. Do not store credentials in Drive files or notebook cells.

In [37]:
import psutil

vm = psutil.virtual_memory()
disk = psutil.disk_usage("/")
try:
    import torch
    gpu_available = torch.cuda.is_available()
    gpu_name = torch.cuda.get_device_name(0) if gpu_available else "None"
except Exception:
    gpu_available, gpu_name = False, "Not checked"

runtime_report = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "cpu_count": os.cpu_count(),
    "ram_total_gb": round(vm.total / (1024**3), 2),
    "ram_available_gb": round(vm.available / (1024**3), 2),
    "disk_free_gb": round(disk.free / (1024**3), 2),
    "gpu_available": gpu_available,
    "gpu_name": gpu_name,
}
print(json.dumps(runtime_report, indent=2))

{
  "python": "3.13.16",
  "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
  "cpu_count": 2,
  "ram_total_gb": 12.67,
  "ram_available_gb": 10.71,
  "disk_free_gb": 112.33,
  "gpu_available": true,
  "gpu_name": "Tesla T4"
}


### 3.1 Optional Google Drive mount

Mount Drive only if you want results to persist across Colab runtime resets. The notebook will create a dedicated project folder. If you choose not to mount Drive, files remain in the temporary Colab runtime and must be downloaded before disconnecting.

In [38]:
USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        PROJECT_DIR = Path("/content/drive/MyDrive/ML_Driven_NoSQL_RAG_Research")
        print("Google Drive mounted.")
    except Exception as exc:
        print("Drive mount unavailable; using temporary runtime storage:", repr(exc))
        PROJECT_DIR = Path("/content/ML_Driven_NoSQL_RAG_Research")
else:
    PROJECT_DIR = Path("/content/ML_Driven_NoSQL_RAG_Research")

DIRS = {
    "raw": PROJECT_DIR / "datasets" / "raw",
    "processed": PROJECT_DIR / "datasets" / "processed",
    "models": PROJECT_DIR / "models",
    "figures": PROJECT_DIR / "reports" / "figures",
    "tables": PROJECT_DIR / "reports" / "tables",
    "logs": PROJECT_DIR / "experiments" / "logs",
    "configs": PROJECT_DIR / "experiments" / "configs",
    "external": PROJECT_DIR / "datasets" / "external",
}
for path in DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
for name, path in DIRS.items():
    print(f"{name:12s} -> {path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted.
Project directory: /content/drive/MyDrive/ML_Driven_NoSQL_RAG_Research
raw          -> /content/drive/MyDrive/ML_Driven_NoSQL_RAG_Research/datasets/raw
processed    -> /content/drive/MyDrive/ML_Driven_NoSQL_RAG_Research/datasets/processed
models       -> /content/drive/MyDrive/ML_Driven_NoSQL_RAG_Research/models
figures      -> /content/drive/MyDrive/ML_Driven_NoSQL_RAG_Research/reports/figures
tables       -> /content/drive/MyDrive/ML_Driven_NoSQL_RAG_Research/reports/tables
logs         -> /content/drive/MyDrive/ML_Driven_NoSQL_RAG_Research/experiments/logs
configs      -> /content/drive/MyDrive/ML_Driven_NoSQL_RAG_Research/experiments/configs
external     -> /content/drive/MyDrive/ML_Driven_NoSQL_RAG_Research/datasets/external


## 4. Install Python dependencies and nosql-rag-research package

This section bootstraps the Python environment in Google Colab.
- If `/content/nosql-rag-research` does not exist, the repository is freshly cloned from `origin/main`.
- If `/content/nosql-rag-research` already exists, it executes `git fetch origin` followed by `git pull --ff-only origin main` to guarantee the latest code is active.
- Verifies and prints the checked-out commit SHA and latest commit message.
- Supports both public repositories (HTTPS) and private repositories via `GITHUB_TOKEN` from Colab Secrets using an ephemeral `GIT_ASKPASS` helper (never printing or saving credentials).
- Dependencies are installed using editable mode: `pip install -q -e ".[all]"`.

> [!IMPORTANT]
> **Colab Notebook Reloading Note:**
> An already-open Google Colab browser tab does not automatically pull updates made to the `.ipynb` notebook file on GitHub.
> - To sync Python code and package modules: re-running Cell 10 pulls the latest changes and reinstalls the package.
> - To reload newly added or modified notebook cells: in Google Colab, use **File -> Open notebook -> GitHub** to reload `NoSQL_RAG_Implementation_Colab.ipynb` from `origin/main`, or refresh your browser tab.

In [39]:
# Securely bootstrap repository and install package in Google Colab
import os, sys, subprocess, shutil, stat
from pathlib import Path

REPO_URL = "https://github.com/hitenBhurani07/nosql-rag-research.git"
TARGET_DIR = Path("/content/nosql-rag-research")

# Check Colab Secrets for optional GITHUB_TOKEN (private repo access)
token = None
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass

git_env = os.environ.copy()
askpass_path = Path("/content/git_askpass.py")
if token:
    print("Detected GITHUB_TOKEN in Colab Secrets. Using secure GIT_ASKPASS helper.")
    askpass_path.write_text('#!/usr/bin/env python3\nimport os, sys\nprint(os.environ.get("GIT_AUTH_TOKEN", ""))\n')
    os.chmod(askpass_path, stat.S_IRUSR | stat.S_IXUSR)
    git_env["GIT_ASKPASS"] = str(askpass_path)
    git_env["GIT_AUTH_TOKEN"] = token
else:
    print("No GITHUB_TOKEN detected; accessing public repository via HTTPS...")

try:
    if not (TARGET_DIR / ".git").exists():
        print(f"Cloning repository from GitHub main branch into {TARGET_DIR}...")
        subprocess.run(["git", "clone", REPO_URL, str(TARGET_DIR)], env=git_env, check=True)
    else:
        print(f"Repository already present at {TARGET_DIR}. Synchronizing with latest origin/main...")
        subprocess.run(["git", "fetch", "origin"], cwd=str(TARGET_DIR), env=git_env, check=True)
        subprocess.run(["git", "checkout", "main"], cwd=str(TARGET_DIR), env=git_env, check=True)
        subprocess.run(["git", "pull", "--ff-only", "origin", "main"], cwd=str(TARGET_DIR), env=git_env, check=True)
finally:
    askpass_path.unlink(missing_ok=True)

# Verify and display the checked-out commit SHA
sha_res = subprocess.run(["git", "rev-parse", "HEAD"], cwd=str(TARGET_DIR), capture_output=True, text=True, check=True)
current_sha = sha_res.stdout.strip()
log_res = subprocess.run(["git", "log", "-1", "--format=%s (%ci)"], cwd=str(TARGET_DIR), capture_output=True, text=True, check=True)
commit_info = log_res.stdout.strip()
print(f"Checked-out commit SHA : {current_sha}")
print(f"Latest commit message  : {commit_info}")

# Change directory to repository root
os.chdir(TARGET_DIR)
if str(TARGET_DIR / "src") not in sys.path:
    sys.path.insert(0, str(TARGET_DIR / "src"))

# Install package and dependencies in editable mode
!pip install -q -e ".[all]"
print(f"Package nosql-rag-research installed successfully at commit {current_sha[:7]}.")

No GITHUB_TOKEN detected; accessing public repository via HTTPS...
Repository already present at /content/nosql-rag-research. Synchronizing with latest origin/main...
Checked-out commit SHA : 89a5cbdd210e330910db43991af1b19a521309dd
Latest commit message  : Created using Colab (2026-11-13 12:48:34 +0530)
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for nosql-rag-research (pyproject.toml) ... done
Package nosql-rag-research installed successfully at commit 89a5cbd.


In [40]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, psutil
from tqdm.auto import tqdm
from IPython.display import display

# Verify core package imports across all research subsystems
import nosql_rag
from nosql_rag.config import DatabaseConfig, ProjectPaths, load_experiment_config
from nosql_rag.workloads import evaluate_feasibility, generate_workload_profiles, make_payload
from nosql_rag.adapters import build_crud_adapter, check_crud_connections, check_qdrant_connection
from nosql_rag.benchmarking import benchmark_common_crud, run_crud_matrix
from nosql_rag.data import DataProvenance, aggregate_crud_runs, derive_best_store_labels, validate_raw_runs, generate_synthetic_demo_runs
from nosql_rag.features import DEFAULT_FEATURE_COLUMNS, grouped_train_test_split, prepare_ml_dataset
from nosql_rag.models import recommend_database, train_and_compare_models
from nosql_rag.explainability import explain_instance_with_shap, explain_model_with_shap
from nosql_rag.evaluation import calculate_latency_regret, summarize_regret
from nosql_rag.rag import run_qdrant_ann_benchmark, run_beir_scifact_rag, evaluate_ranked_results

print("✅ nosql_rag version:", nosql_rag.__version__)
print("✅ All package submodules imported cleanly without errors.")

RUNTIME_PACKAGES = {
    "nosql_rag": nosql_rag.__version__,
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": __import__("sklearn").__version__,
    "matplotlib": __import__("matplotlib").__version__,
}
print(json.dumps(RUNTIME_PACKAGES, indent=2))


✅ nosql_rag version: 0.1.0
✅ All package submodules imported cleanly without errors.
{
  "nosql_rag": "0.1.0",
  "python": "3.13.16",
  "numpy": "2.4.6",
  "pandas": "2.2.3",
  "sklearn": "1.6.1",
  "matplotlib": "3.10.0"
}


### 5.1 Discover Google Colab Public IP for MongoDB Atlas Allowlisting
Google Colab runtimes execute on ephemeral Google Cloud VMs with dynamic egress IPs. Before connecting to MongoDB Atlas M0, you must allowlist the runtime's public IPv4 address.
> [!IMPORTANT]
> In MongoDB Atlas: **Security → Network Access → IP Access List → Add IP Address**. Add the IP below as `<COLAB_PUBLIC_IP>/32` with an expiration timer (e.g. 1 hour or 6 hours). Never use `0.0.0.0/0`. If Colab restarts or reconnects, re-run this cell to check if your IP changed.

In [41]:
# Discover current Colab public IPv4 address for MongoDB Atlas Network Access
import requests

try:
    colab_public_ip = requests.get("https://api.ipify.org", timeout=15).text.strip()
    print("=" * 60)
    print("  Google Colab Runtime Egress IPv4 Address")
    print("=" * 60)
    print(f"👉 Current Colab Public IP: {colab_public_ip}")
    print("\nNext step in MongoDB Atlas:")
    print("1. Navigate to: Security → Network Access → IP Access List")
    print(f"2. Click 'Add IP Address' and enter: {colab_public_ip}/32")
    print("3. (Recommended) Set entry to expire in 1 hour or 6 hours")
    print("=" * 60)
except Exception as exc:
    print("⚠️ Could not determine Colab public IP:", exc)

  Google Colab Runtime Egress IPv4 Address
👉 Current Colab Public IP: 136.85.69.137

Next step in MongoDB Atlas:
1. Navigate to: Security → Network Access → IP Access List
2. Click 'Add IP Address' and enter: 136.85.69.137/32
3. (Recommended) Set entry to expire in 1 hour or 6 hours


### 5.2 Stage DataStax Astra DB Secure Connect Bundle
DataStax Astra DB Serverless mandates mutual TLS (mTLS) via a downloaded Secure Connect Bundle (`.zip`) archive.
Upload your downloaded bundle (e.g., `secure-connect-nosql-rag-research.zip`). This cell will safely stage it into `/tmp/astra_scb/secure-connect-bundle.zip` with restricted file permissions (`0600`).
> [!NOTE]
> The bundle is staged outside the project workspace and Google Drive to ensure it is never committed to GitHub or packaged into experiment archives. `/tmp` is ephemeral; if the Colab runtime restarts, re-run this cell to re-stage your bundle.

In [42]:
# Stage DataStax Astra DB Secure Connect Bundle into restricted temporary storage
import os
import shutil
import zipfile
from pathlib import Path

SCB_DIR = Path("/tmp/astra_scb")
SCB_TARGET = SCB_DIR / "secure-connect-bundle.zip"
EXPECTED_ASTRA_ENTRIES = {"config.json", "cqlshrc", "cert", "key", "truststore.jks", "ca.crt"}

SCB_DIR.mkdir(parents=True, exist_ok=True)
os.chmod(SCB_DIR, 0o700)

already_staged = False
if SCB_TARGET.is_file():
    try:
        with zipfile.ZipFile(SCB_TARGET, "r") as zf:
            namelist = set(zf.namelist())
            if namelist & EXPECTED_ASTRA_ENTRIES or any("cert" in n.lower() or "cqlsh" in n.lower() for n in namelist):
                already_staged = True
                print(f"ℹ️ Secure Connect Bundle already staged and validated at: {SCB_TARGET}")
    except Exception:
        already_staged = False

if not already_staged:
    print("Please select and upload your Astra Secure Connect Bundle (e.g. secure-connect-nosql-rag-research.zip)...")
    uploaded = {}
    try:
        from google.colab import files
        uploaded = files.upload()
    except ImportError:
        # Local development / non-Colab fallback
        for p in Path("/content").glob("secure-connect-*.zip"):
            uploaded[p.name] = p.read_bytes()
            break

    if not uploaded:
        print("⚠️ No bundle uploaded. Run this cell and select your downloaded Astra .zip file.")
    elif len(uploaded) > 1:
        print("⚠️ Multiple files uploaded. Please upload exactly one Astra Secure Connect Bundle .zip file.")
    else:
        uploaded_name = list(uploaded.keys())[0]
        if not uploaded_name.endswith(".zip"):
            print(f"⚠️ Uploaded file '{uploaded_name}' is not a .zip archive. Please upload the genuine Astra bundle.")
        else:
            SCB_TARGET.write_bytes(uploaded[uploaded_name])
            os.chmod(SCB_TARGET, 0o600)

            # Clean up uploaded file from notebook working directory
            if Path(uploaded_name).exists() and Path(uploaded_name).resolve() != SCB_TARGET.resolve():
                Path(uploaded_name).unlink(missing_ok=True)

            try:
                with zipfile.ZipFile(SCB_TARGET, "r") as zf:
                    corrupt = zf.testzip()
                    if corrupt is not None:
                        raise ValueError(f"Corrupt ZIP entry detected: {corrupt}")
                    namelist = set(zf.namelist())
                    if not (namelist & EXPECTED_ASTRA_ENTRIES or any("cert" in n.lower() or "cqlsh" in n.lower() for n in namelist)):
                        print("⚠️ Warning: Archive does not contain standard Astra markers (config.json, cqlshrc, cert).")
                    else:
                        print(f"✅ Astra Secure Connect Bundle successfully staged and verified at: {SCB_TARGET}")
                        print("🔒 File permissions set to 0600. Isolated from Git working tree and Drive exports.")
            except Exception as exc:
                print(f"❌ Failed to validate Secure Connect Bundle ZIP: {type(exc).__name__}: {exc}")
                SCB_TARGET.unlink(missing_ok=True)

ℹ️ Secure Connect Bundle already staged and validated at: /tmp/astra_scb/secure-connect-bundle.zip


## 5.3 Configure Database Credentials Using Colab Secrets

Open the **key icon (Secrets)** in the Google Colab left sidebar and add each secret listed below. Ensure the **Notebook access** toggle switch is enabled for each key. Never enter passwords or raw credentials directly into notebook code cells.

### Required Secrets for Provisioned Databases

| Secret | Purpose | Target Provider / Provisioned Resource |
|---|---|---|
| `REDIS_URL` | Redis Cloud connection URI | Redis Cloud (`profit-cook-stitch-74158.db.redis.io:18690`) |
| `MONGODB_URI` | MongoDB Atlas connection URI | MongoDB Atlas M0 (`cluster0.hqd8wfk.mongodb.net`) |
| `MONGODB_DATABASE` | Database name | `nosql_rag_research` |
| `CASSANDRA_MODE` | Cassandra connection mode | `astra` |
| `CASSANDRA_KEYSPACE` | CQL Keyspace | `nosql_rag_research` |
| `CASSANDRA_APPLICATION_TOKEN` | Astra database application token | DataStax Astra DB (`AstraCS:...`) |
| `CASSANDRA_SECURE_CONNECT_BUNDLE_PATH` | Staged Secure Connect Bundle path | `/tmp/astra_scb/secure-connect-bundle.zip` |

### Step-by-Step Instructions:
1. Open the Google Colab Secrets sidebar using the key icon on the left.
2. Add each secret with the exact spelling shown above.
3. Enable Notebook access for each secret.
4. Never enter actual passwords into notebook code cells.

### Unprovisioned Services (Safely Skipped)
`NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`, `QDRANT_URL`, and `QDRANT_API_KEY` are not yet provisioned. When omitted, connection checks will cleanly report **`SKIPPED`** without causing failures.

In [43]:
# Validate runtime configuration presence without leaking secrets
from nosql_rag.config import DatabaseConfig, validate_runtime_configuration

db_config = DatabaseConfig.from_env()

# Run structural and endpoint validation
validations = validate_runtime_configuration(db_config)
validation_df = pd.DataFrame(validations)
display(validation_df[["database", "status", "configured", "valid", "details"]])

print("\n--- Configuration Status Overview ---")
for item in validations:
    status_icon = "✅" if item["status"] == "configured" else ("⏭️" if "skipped" in item["status"] else "⚠️")
    print(f"{status_icon} {item['database']:<12}: {item['status']}")
print("-------------------------------------")
print("🔒 Secret contents, passwords, and tokens are intentionally never displayed.")

,database,status,configured,valid,details
0,Redis,configured,True,True,Valid redis:// endpoint (profit-cook-stitch-74...
1,MongoDB,configured,True,True,Valid mongodb+srv:// cluster (cluster0.hqd8wfk...
2,Cassandra,configured,True,True,Astra mode validated (keyspace: nosql_rag_rese...
3,Neo4j,configured,True,True,Configured
4,Qdrant,configured,True,True,Configured



--- Configuration Status Overview ---
✅ Redis       : configured
✅ MongoDB     : configured
✅ Cassandra   : configured
✅ Neo4j       : configured
✅ Qdrant      : configured
-------------------------------------
🔒 Secret contents, passwords, and tokens are intentionally never displayed.


## 6. Experiment configuration and compatibility policy

The default ML training task uses a common logical record CRUD workload. It compares Redis, MongoDB, Cassandra and Neo4j. Qdrant is handled in the separate vector retrieval experiment because a native ANN search is not semantically comparable to a key-value record read.

This is a deliberate research control, not an accidental omission. A later five-way vector selector requires additional vector-capable adapters and compatible vector indexes for the other engines. Record any such extensions in the project decision log.

In [44]:
CANDIDATE_CRUD_ENGINES = ["Redis", "MongoDB", "Cassandra", "Neo4j"]
VECTOR_ENGINE = "Qdrant"
DATABASE_HOSTING_REGION = "unknown - fill this in before final experiments"
DATABASE_DEPLOYMENT_NOTES = "fill in provider, region, instance size, TLS mode and relevant network path"

COMPATIBILITY_MATRIX = pd.DataFrame([
    {"workload_family": "key_value_crud", "database": "Redis", "included_in_default_ml": True, "notes": "Native key/value record operations"},
    {"workload_family": "key_value_crud", "database": "MongoDB", "included_in_default_ml": True, "notes": "Indexed _id document lookup/update"},
    {"workload_family": "key_value_crud", "database": "Cassandra", "included_in_default_ml": True, "notes": "Partition-key lookup/update"},
    {"workload_family": "key_value_crud", "database": "Neo4j", "included_in_default_ml": True, "notes": "Indexed property lookup/update on nodes"},
    {"workload_family": "vector_ann", "database": "Qdrant", "included_in_default_ml": False, "notes": "Separate vector quality/latency evaluation"},
])
display(COMPATIBILITY_MATRIX)

,workload_family,database,included_in_default_ml,notes
0,key_value_crud,Redis,True,Native key/value record operations
1,key_value_crud,MongoDB,True,Indexed _id document lookup/update
2,key_value_crud,Cassandra,True,Partition-key lookup/update
3,key_value_crud,Neo4j,True,Indexed property lookup/update on nodes
4,vector_ann,Qdrant,False,Separate vector quality/latency evaluation


## 7. Logging and experiment metadata

Every run receives a unique run identifier and timestamp. Raw run-level measurements are preserved. Aggregation happens later and must never overwrite raw results.

In [45]:
import logging

LOG_FILE = DIRS["logs"] / "colab_experiment.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.FileHandler(LOG_FILE), logging.StreamHandler()]
)
logger = logging.getLogger("nosql_rag_research")

EXPERIMENT_CONTEXT = {
    "experiment_id": str(uuid.uuid4()),
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "client_platform": platform.platform(),
    "client_python": sys.version.split()[0],
    "client_cpu_count": os.cpu_count(),
    "client_ram_gb": round(psutil.virtual_memory().total / (1024**3), 2),
    "client_location": "Colab-managed runtime; exact region not assumed",
    "primary_objective": PRIMARY_OBJECTIVE,
    "random_seed": RANDOM_SEED,
}

with open(PROJECT_DIR / "experiment_context.json", "w") as f:
    json.dump(EXPERIMENT_CONTEXT, f, indent=2)

print(json.dumps(EXPERIMENT_CONTEXT, indent=2))

{
  "experiment_id": "804e2dcf-61d8-4ccb-aa56-b756d64fca96",
  "created_utc": "2026-11-13T07:24:01.998479+00:00",
  "client_platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
  "client_python": "3.13.16",
  "client_cpu_count": 2,
  "client_ram_gb": 12.67,
  "client_location": "Colab-managed runtime; exact region not assumed",
  "primary_objective": "min_p95_latency_ms",
  "random_seed": 42
}


## 8. Database adapters

The following adapters implement a deliberately narrow, comparable logical workload: seed records, read a record by ID, update a record by ID, and clean up the temporary namespace. Database-specific schema/index choices are recorded in the run metadata.

The benchmark uses one client operation per measured request. Concurrency is controlled by a thread pool. A failed operation is retained as an error; it is not converted into a zero-latency success.

In [46]:
from nosql_rag.adapters import (
    AdapterError,
    BaseAdapter,
    BaseCRUDAdapter,
    CassandraAdapter,
    MongoAdapter,
    Neo4jAdapter,
    QdrantAdapter,
    RedisAdapter,
    build_crud_adapter,
    check_crud_connections,
    check_qdrant_connection,
)
print('Imported database adapters and connection checkers from nosql_rag.adapters.')


Imported database adapters and connection checkers from nosql_rag.adapters.


In [47]:
# Adapters are implemented in nosql_rag.adapters with native clients for Redis, MongoDB, Cassandra, Neo4j, and Qdrant.
print('Using production adapter implementations from nosql_rag.adapters.')


Using production adapter implementations from nosql_rag.adapters.


In [48]:
# Execute live connection checks across candidate databases
if RUN_CONNECTION_CHECKS:
    crud_results = check_crud_connections(db_config)
    qdrant_res = check_qdrant_connection(db_config)
    all_results = crud_results + [qdrant_res]

    print("\n" + "=" * 65)
    print("  NoSQL RAG Research — Live Connection Verification")
    print("=" * 65)

    connected_count = 0
    failed_count = 0
    skipped_count = 0

    for r in all_results:
        db = r["database"].upper()
        status = r.get("status", "CONNECTED" if r["connected"] else "FAILED")
        error = r.get("error")
        latency = f"{r['connect_check_ms']:.1f}ms" if r.get("connect_check_ms") is not None else "N/A"
        version = r.get("version") or "unknown"

        if status == "CONNECTED":
            connected_count += 1
            print(f"✅ {db:<12} CONNECTED (version: {version}, ping: {latency})")
        elif status == "SKIPPED":
            skipped_count += 1
            print(f"⏭️ {db:<12} SKIPPED ({error})")
        else:
            failed_count += 1
            print(f"❌ {db:<12} FAILED ({error})")

    print("=" * 65)
    print("Connection Summary:")
    print("Required initial integrations: 3 (Redis, MongoDB, Cassandra/Astra)")
    print(f"Successful live connections:   {connected_count}")
    print(f"Failed live connections:       {failed_count}")
    print(f"Skipped unconfigured services: {skipped_count}")
    print("=" * 65 + "\n")

    connection_results = pd.DataFrame(all_results)
    display(connection_results[["database", "status", "connected", "version", "connect_check_ms", "error"]])
    connection_results.to_csv(DIRS['raw'] / 'connection_checks.csv', index=False)
    print(f"Saved connection check results to: {DIRS['raw'] / 'connection_checks.csv'}")
else:
    print("Connection checks disabled. Set RUN_CONNECTION_CHECKS=True when secrets are configured.")

/usr/local/lib/python3.13/dist-packages/neo4j/_sync/work/result.py:631: UserWarning: Expected a result with a single record, but found multiple.
  warn(



  NoSQL RAG Research — Live Connection Verification
✅ REDIS        CONNECTED (version: 8.8.0, ping: 460.3ms)
✅ MONGODB      CONNECTED (version: 8.0.37, ping: 761.9ms)
✅ CASSANDRA    CONNECTED (version: 4.0.11.0-6819aa9c7f6c, ping: 12509.0ms)
✅ NEO4J        CONNECTED (version: 5.27-aura, ping: 192.1ms)
✅ QDRANT       CONNECTED (version: qdrant-remote, ping: 2031.8ms)
Connection Summary:
Required initial integrations: 3 (Redis, MongoDB, Cassandra/Astra)
Successful live connections:   5
Failed live connections:       0
Skipped unconfigured services: 0



,database,status,connected,version,connect_check_ms,error
0,Redis,CONNECTED,True,8.8.0,460.290,None
1,MongoDB,CONNECTED,True,8.0.37,761.915,None
2,Cassandra,CONNECTED,True,4.0.11.0-6819aa9c7f6c,12509.025,None
3,Neo4j,CONNECTED,True,5.27-aura,192.059,None
4,Qdrant,CONNECTED,True,qdrant-remote,2031.823,None


Saved connection check results to: /content/drive/MyDrive/ML_Driven_NoSQL_RAG_Research/datasets/raw/connection_checks.csv


### 8.1 Database-Independent Smoke Test
This cell verifies the end-to-end mechanics of the repository (CLI, configuration, synthetic data generation, dataset validation, and Drive persistence) without requiring any live database endpoints or credentials.
> [!NOTE]
> Synthetic measurements generated here carry `data_provenance=synthetic_demo` and are never used to train the research model.


In [49]:
# Database-Independent Smoke Test (Safe, offline verification of pipeline & Drive persistence)
if RUN_SMOKE_TEST:
    print("--- Starting Database-Independent Smoke Test ---")
    # 1. Verify CLI
    cli_out = subprocess.check_output([sys.executable, "-m", "nosql_rag.cli", "--help"], text=True)
    assert "nosql-rag" in cli_out, "CLI help test failed"
    print("✅ CLI check: Passed")

    # 2. Verify Config Loading
    exp_cfg = load_experiment_config()
    assert exp_cfg.primary_objective == "min_p95_latency_ms"
    print(f"✅ Config loading: Passed ({exp_cfg.primary_objective})")

    # 3. Verify Workload Generator & Feasibility Filter
    smoke_profs = generate_workload_profiles(n_profiles=4, seed=RANDOM_SEED, id_prefix="SMOKE")
    assert len(smoke_profs) == 4
    feas = evaluate_feasibility({"query_type": "vector_similarity", "vector_dimensions": 128})
    assert feas.feasible_databases == ["Qdrant"]
    print("✅ Workload generator & feasibility engine: Passed")

    # 4. Generate Synthetic Demo Data (Clearly tagged synthetic_demo)
    syn_runs = generate_synthetic_demo_runs(smoke_profs, repetitions=2, seed=RANDOM_SEED)
    assert (syn_runs["data_provenance"] == DataProvenance.SYNTHETIC_DEMO.value).all()
    syn_path = DIRS["raw"] / "smoke_test_synthetic_runs.csv"
    syn_runs.to_csv(syn_path, index=False)
    print(f"✅ Synthetic demo persistence: Passed ({len(syn_runs)} rows -> {syn_path})")

    # 5. Verify Dataset Validator
    report = validate_raw_runs(syn_runs, profiles=smoke_profs, allow_synthetic=True)
    assert report.is_valid, f"Validation failed: {report.errors}"
    print("✅ Dataset validation check: Passed")
    print("--- Smoke Test Completed Successfully (Zero live database credentials needed) ---\n")
else:
    print("Smoke test disabled. Set RUN_SMOKE_TEST=True to verify pipeline offline.")


--- Starting Database-Independent Smoke Test ---
✅ CLI check: Passed
✅ Config loading: Passed (min_p95_latency_ms)
✅ Workload generator & feasibility engine: Passed
✅ Synthetic demo persistence: Passed (32 rows -> /content/drive/MyDrive/ML_Driven_NoSQL_RAG_Research/datasets/raw/smoke_test_synthetic_runs.csv)
✅ Dataset validation check: Passed
--- Smoke Test Completed Successfully (Zero live database credentials needed) ---



## 9. Generate reproducible workload profiles

The initial ML pilot covers two common logical record workloads: `key_value_crud` and `document_crud`. Both map to the same read/update semantics and payload, but the adapters use each database's native storage representation.

The consistency requirement is stored as metadata but is not varied in the default pilot. It is intentionally excluded from the initial ML feature set until each engine's consistency policy is explicitly configured and verified.

In [50]:
from nosql_rag.workloads import generate_workload_profiles

pilot_profiles = generate_workload_profiles(
    n_profiles=PILOT_WORKLOAD_PROFILES,
    operations=PILOT_OPERATIONS,
    records=PILOT_RECORDS,
    concurrency_levels=[PILOT_CONCURRENCY],
    id_prefix='P',
)
full_profiles = generate_workload_profiles(
    n_profiles=FULL_WORKLOAD_PROFILES,
    operations=FULL_OPERATIONS_PER_RUN,
    records=FULL_RECORDS_PER_RUN,
    concurrency_levels=FULL_CONCURRENCY_LEVELS,
    id_prefix='F',
)

display(pilot_profiles)
print('Pilot workload profiles:', len(pilot_profiles))
print('Full planned workload profiles:', len(full_profiles))


,workload_id,query_type,read_ratio,write_ratio,payload_bytes,dataset_records,data_size_mb,data_size_gb,vector_dimensions,top_k,graph_depth,consistency_requirement,concurrency,n_operations,random_seed
0,P0001,key_value_crud,0.2,0.8,8192,200,1.56250,0.001526,0,0,0,default,1,250,42
1,P0002,document_crud,0.5,0.5,8192,100,0.78125,0.000763,0,0,0,default,1,500,43


Pilot workload profiles: 2
Full planned workload profiles: 75


## 10. Run the common CRUD benchmark

### Benchmark protocol

For each workload/database pair:

1. Create a run-specific namespace/table/collection.
2. Seed the requested number of records outside the timed measurement window.
3. Generate a deterministic operation sequence from the workload's read/write ratios.
4. Execute the sequence with the requested concurrency.
5. Record per-operation latency, total elapsed time, errors and client-side resource measurements.
6. Calculate p50/p95 latency and operations per second.
7. Clean up temporary records by default.

This benchmark measures client-observed operation latency from the Colab runtime. It does not measure database-server CPU/RAM unless server monitoring is added separately.

In [51]:
from nosql_rag.benchmarking import benchmark_common_crud, run_crud_matrix
print('CRUD benchmarking engine loaded from nosql_rag.benchmarking.')


CRUD benchmarking engine loaded from nosql_rag.benchmarking.


## 11. Save and resume benchmark runs

Every run is appended to a raw CSV. The CSV is a run log, not an aggregated model dataset. This allows a Colab runtime reset to be handled without losing already-saved batches.

In [52]:
from nosql_rag.data import append_raw_runs, load_raw_runs

CRUD_RUNS_FILE = DIRS['raw'] / 'crud_benchmark_runs.csv'
print('Run persistence bound to:', CRUD_RUNS_FILE)


Run persistence bound to: /content/drive/MyDrive/ML_Driven_NoSQL_RAG_Research/datasets/raw/crud_benchmark_runs.csv


## 12. Full benchmark matrix (run in controlled batches)

The proposed target is 75 unique workload profiles × 4 CRUD-compatible database engines = 300 unique workload/database configurations before repetitions. Three repetitions produce 900 run-level measurement rows if every configuration completes.

This is a planning target only. A configuration counts only after a real database experiment has completed and is recorded. Run a small batch, inspect stability, then continue. Do not leave `RUN_FULL_MATRIX=True` until the pilot has passed.

In [53]:
from nosql_rag.benchmarking import run_crud_matrix

if RUN_FULL_MATRIX:
    print('Starting full benchmark matrix execution...')
    full_runs = run_crud_matrix(
        profiles=full_profiles,
        databases=CANDIDATE_CRUD_ENGINES,
        db_config=db_config,
        output_csv=CRUD_RUNS_FILE,
        repetitions=FULL_REPETITIONS,
        batch_size=FULL_BATCH_SIZE,
        warmup_operations=20,
        cleanup=CLEANUP_AFTER_RUN,
        timeout_seconds=300,
    )
    print(f'Full matrix runs completed. Total rows in {CRUD_RUNS_FILE}: {len(full_runs)}')
else:
    print('Full benchmark matrix is OFF. Set RUN_FULL_MATRIX=True when ready for complete execution.')


Full benchmark matrix is OFF. Set RUN_FULL_MATRIX=True when ready for complete execution.


## 13. Inspect benchmark data quality

This section checks whether results are genuine recorded runs, complete and usable. A partial or failed run is not silently discarded from the audit trail.

In [54]:
from nosql_rag.data import load_raw_runs, validate_raw_runs

crud_runs = load_raw_runs(CRUD_RUNS_FILE)
if crud_runs.empty:
    print('No benchmark measurements found yet. Run the pilot after configuring database secrets.')
else:
    report = validate_raw_runs(crud_runs, profiles=pd.concat([pilot_profiles, full_profiles]), allow_synthetic=False)
    print(f'Data Quality Valid: {report.is_valid} | Valid Successful Runs: {report.valid_rows}/{report.total_rows}')
    display(crud_runs['status'].value_counts(dropna=False).rename_axis('status').reset_index(name='run_count'))
    if report.errors:
        print('Integrity Errors:', report.errors)
    if report.warnings:
        print('Integrity Warnings:', report.warnings[:5])


No benchmark measurements found yet. Run the pilot after configuring database secrets.


## 14. Aggregate repetitions and derive best-store labels

The default primary label policy is the database with the lowest median p95 latency for a workload, among databases with successful runs and an error rate at or below the predeclared threshold. The comparison is restricted to databases measured for the same workload profile.

This is a single-objective classifier label. It does not mean the selected database is best on every dimension. Throughput, resource use and other trade-offs remain in the result tables.

In [55]:
from nosql_rag.data import aggregate_crud_runs, derive_best_store_labels, validate_raw_runs

aggregated_results = aggregate_crud_runs(crud_runs) if not crud_runs.empty else pd.DataFrame()
if not aggregated_results.empty:
    display(aggregated_results.head())
    observed_ids = set(aggregated_results['workload_id'])
    known_profiles = pd.concat([pilot_profiles, full_profiles]).drop_duplicates('workload_id')
    used_profiles = known_profiles[known_profiles['workload_id'].isin(observed_ids)]
    labelled_dataset = derive_best_store_labels(aggregated_results, used_profiles, objective=PRIMARY_OBJECTIVE)
    if not labelled_dataset.empty:
        labelled_dataset.to_csv(DIRS['processed'] / 'labelled_workloads.csv', index=False)
        aggregated_results.to_csv(DIRS['processed'] / 'aggregated_crud_measurements.csv', index=False)
        print('Labelled workload rows:', len(labelled_dataset))
        display(labelled_dataset['best_store'].value_counts().rename_axis('best_store').reset_index(name='workload_count'))
    else:
        print('No workload had at least two successful eligible database measurements.')
else:
    labelled_dataset = pd.DataFrame()
    print('No valid aggregated benchmark results available yet.')


No valid aggregated benchmark results available yet.


## 15. Prepare model features and train/test split

Only workload features that were actually varied and recorded in the default CRUD experiment are used. The initial feature set excludes consistency and vector dimensions because those are not varied by the common CRUD adapters. They remain in the broader research schema for future controlled experiments.

The target is the measured best-store label. Do not add p95 latency, throughput or any derived label metric to the model input: that would leak the answer into the features.

In [56]:
from nosql_rag.features import DEFAULT_FEATURE_COLUMNS, grouped_train_test_split, prepare_ml_dataset

if not labelled_dataset.empty:
    model_data, X, y, groups = prepare_ml_dataset(labelled_dataset, feature_columns=FEATURE_COLUMNS)
    class_counts = y.value_counts()
    display(class_counts.rename_axis('class').reset_index(name='count'))
    split = grouped_train_test_split(X, y, groups, model_data, test_size=0.25, random_state=RANDOM_SEED)
    X_train, X_test = split['X_train'], split['X_test']
    y_train, y_test = split['y_train'], split['y_test']
    train_meta, test_meta = split['meta_train'], split['meta_test']
    print('Training workloads:', len(X_train), '| Test workloads:', len(X_test))
else:
    print('ML preparation skipped until measured labels exist.')


ML preparation skipped until measured labels exist.


## 16. Train and compare the three required models

A one-hot encoder handles categorical fields. Median imputation and scaling are applied to numeric fields. The same split and feature pipeline are used for all models. The final model is chosen using held-out Macro-F1, not training accuracy.

In [57]:
from nosql_rag.models import train_and_compare_models

if RUN_ML_TRAINING and not labelled_dataset.empty:
    fitted_models, model_predictions, model_metrics, best_model_name = train_and_compare_models(
        X_train, X_test, y_train, y_test, random_state=RANDOM_SEED
    )
    best_model = fitted_models[best_model_name]
    display(model_metrics)
    model_metrics.to_csv(DIRS['tables'] / 'model_comparison.csv', index=False)
    print('Selected model by held-out Macro-F1:', best_model_name)
else:
    fitted_models, model_predictions, model_metrics, best_model_name, best_model = {}, {}, pd.DataFrame(), None, None
    print('Model training is OFF or no measured labelled dataset exists. Set RUN_ML_TRAINING=True after validating labels.')


Model training is OFF or no measured labelled dataset exists. Set RUN_ML_TRAINING=True after validating labels.


## 17. Model evaluation plots and classification report

In [58]:
if best_model is not None:
    display(pd.DataFrame(classification_report(y_test, model_predictions[best_model_name], output_dict=True, zero_division=0)).T)
    fig, ax=plt.subplots(figsize=(8,6))
    ConfusionMatrixDisplay.from_predictions(y_test, model_predictions[best_model_name], ax=ax, xticks_rotation=35, colorbar=False)
    ax.set_title(f"Held-out confusion matrix — {best_model_name}")
    plt.tight_layout()
    plt.savefig(DIRS["figures"] / "confusion_matrix_best_model.png", dpi=220, bbox_inches="tight")
    plt.show()

    fig, ax=plt.subplots(figsize=(8,4))
    model_metrics.set_index("model")[["accuracy","macro_f1","weighted_f1"]].plot(kind="bar", ax=ax)
    ax.set_ylim(0,1)
    ax.set_ylabel("Score")
    ax.set_title("Held-out model comparison")
    ax.tick_params(axis="x", rotation=20)
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.savefig(DIRS["figures"] / "model_comparison.png", dpi=220, bbox_inches="tight")
    plt.show()
else:
    print("No executed model results to plot.")

No executed model results to plot.


## 18. Save the selected model and its schema

Save the preprocessing pipeline and estimator together so inference uses exactly the same feature transformations as training.

In [59]:
if best_model is not None:
    model_path=DIRS["models"] / "best_nosql_recommender.joblib"
    joblib.dump(best_model, model_path)
    model_metadata={
        "model_name":best_model_name,
        "features":FEATURE_COLUMNS,
        "target":TARGET_COLUMN,
        "selection_metric":"held_out_macro_f1",
        "metrics":model_metrics.to_dict(orient="records"),
        "random_seed":RANDOM_SEED,
        "training_utc":datetime.now(timezone.utc).isoformat(),
        "data_rows":int(len(model_data)),
        "warning":"The model is valid only within the measured workload/database compatibility scope.",
    }
    with open(DIRS["models"] / "model_metadata.json", "w") as f:
        json.dump(model_metadata, f, indent=2)
    print("Saved:", model_path)
else:
    print("No model artifact saved because training has not been executed.")

No model artifact saved because training has not been executed.


## 19. Latency regret against baselines

Relative latency regret for one workload is:

\[
\text{Regret} = \frac{L_{recommended}-L_{best}}{L_{best}}
\]

where `L` is the median measured p95 latency for that workload. A regret of 0 means the recommendation selected the measured best candidate. Regret is calculated only when the selected database has a valid measurement for that workload.

Baselines are declared before inspecting test outcomes:

- **Rule-based:** Redis for key-value CRUD and MongoDB for document CRUD.
- **Majority class:** most frequent training-set best-store label.
- **Single-store default:** MongoDB for every workload.

These are simple baselines, not claims about an optimal architecture.

In [60]:
from nosql_rag.evaluation import calculate_latency_regret, summarize_regret

if best_model is not None and not aggregated_results.empty:
    regret_results = calculate_latency_regret(best_model, X_test, y_test, test_meta, aggregated_results, y_train)
    display(summarize_regret(regret_results))
    regret_results.to_csv(DIRS['tables'] / 'latency_regret_by_workload.csv', index=False)
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.boxplot(data=regret_results, x='method', y='relative_latency_regret', ax=ax)
    ax.set_title('Held-out relative latency regret by selection method')
    ax.tick_params(axis='x', rotation=20)
    plt.tight_layout()
    plt.savefig(DIRS['figures'] / 'latency_regret_comparison.png', dpi=220, bbox_inches='tight')
    plt.show()
else:
    regret_results = pd.DataFrame()
    print('Regret evaluation requires a trained model and measured held-out workload results.')


Regret evaluation requires a trained model and measured held-out workload results.


## 20. Explainability using SHAP

SHAP will explain the selected tree-based model. The explanation is an attribution of model output to input features; it is not proof that a feature causally changes database performance. The encoded feature names are preserved so the plot can be audited.

In [61]:
from nosql_rag.explainability import explain_model_with_shap

if RUN_SHAP and best_model is not None:
    shap_out = DIRS['figures'] / 'shap_summary.png'
    shap_res = explain_model_with_shap(best_model, X_train, X_test, output_figure_path=shap_out)
    display(shap_res.global_importance.head(10))
    print('SHAP summary saved to:', shap_out)
else:
    print('SHAP is OFF or there is no trained model. Set RUN_SHAP=True after ML training.')


SHAP is OFF or there is no trained model. Set RUN_SHAP=True after ML training.


## 21. Make a recommendation for a new workload

This inference helper validates required fields and applies the saved preprocessing pipeline. It does not guarantee performance outside the training distribution. The caller should compare the input against the observed training ranges and display an out-of-range warning.

In [62]:
from nosql_rag.models import recommend_database

if best_model is not None:
    example_input = {
        'query_type': 'key_value_crud',
        'read_ratio': 0.90,
        'write_ratio': 0.10,
        'payload_bytes': 512,
        'dataset_records': 1000,
        'data_size_mb': 1000 * 512 / (1024**2),
        'concurrency': 1,
    }
    example_recommendation = recommend_database(example_input, trained_pipeline=best_model, training_data=model_data)
    print(json.dumps(example_recommendation, indent=2))
else:
    print('Inference unavailable until measured data has trained a model.')


Inference unavailable until measured data has trained a model.


## 22. Qdrant native ANN benchmark

This is a separate vector retrieval experiment. It generates seeded vectors, uploads them to Qdrant, calculates exact nearest neighbours locally as ground truth, measures Qdrant query latency and calculates Recall@K.

The resulting latency is an end-to-end client-observed Qdrant query time from Colab. It must not be compared directly with the common CRUD benchmark.

In [63]:
from nosql_rag.rag import run_qdrant_ann_benchmark

if RUN_QDRANT_ANN_PILOT:
    qdrant_ann_result = run_qdrant_ann_benchmark(db_config)
    display(pd.DataFrame([qdrant_ann_result]))
    with open(DIRS['raw'] / 'qdrant_ann_runs.json', 'w') as f:
        json.dump(qdrant_ann_result, f, indent=2)
else:
    print('Qdrant ANN experiment is OFF. Configure QDRANT_URL and set RUN_QDRANT_ANN_PILOT=True to execute.')


Qdrant ANN experiment is OFF. Configure QDRANT_URL and set RUN_QDRANT_ANN_PILOT=True to execute.


## 23. RAG retrieval validation using BEIR SciFact

This optional section downloads the public BEIR SciFact dataset, embeds corpus documents and queries with a fixed SentenceTransformer model, indexes corpus vectors in Qdrant and evaluates retrieved results against BEIR relevance judgments.

This measures the **retrieval stage** of a RAG pipeline. It does not measure generated-answer faithfulness or LLM answer quality. Those require a separate, controlled generation/evaluation protocol.

In [64]:
from nosql_rag.rag import run_beir_scifact_rag

if RUN_BEIR_RAG:
    beir_result = run_beir_scifact_rag(db_config, max_queries=200)
    display(pd.DataFrame([beir_result]))
    with open(DIRS['raw'] / 'beir_scifact_qdrant_results.json', 'w') as f:
        json.dump(beir_result, f, indent=2)
else:
    print('BEIR RAG retrieval experiment is OFF. Configure QDRANT_URL and set RUN_BEIR_RAG=True to execute.')


BEIR RAG retrieval experiment is OFF. Configure QDRANT_URL and set RUN_BEIR_RAG=True to execute.


## 24. Import benchmark data produced outside Colab (optional)

If the database benchmark runner is executed on a dedicated VM or another controlled host, upload its raw CSV into `datasets/raw/` and load it here. The imported file must follow the same column schema. Never merge data from different hardware/network configurations without retaining those environment identifiers.

In [65]:
def import_external_runs(csv_path):
    frame=pd.read_csv(csv_path)
    required={"run_id","workload_id","database","status","p95_latency_ms","throughput_ops_sec","error_rate"}
    missing=required-set(frame.columns)
    if missing:
        raise ValueError(f"External run file is missing required fields: {sorted(missing)}")
    frame["import_source_path"]=str(csv_path)
    return frame

# Example after uploading a runner output:
# external_runs = import_external_runs(DIRS["raw"] / "external_vm_crud_runs.csv")
# crud_runs = pd.concat([crud_runs, external_runs], ignore_index=True)
print("External-run import helper is ready. No external measurements have been imported.")

External-run import helper is ready. No external measurements have been imported.


## 25. Export project artifacts and create a handoff package

This section gathers available outputs. It does not create missing results. Include a manifest with file sizes and timestamps so a later Colab runtime or a different AI model can identify the latest valid artifacts.

In [66]:
def write_project_status():
    # 1. Use the correct imported load function
    runs = load_raw_runs(CRUD_RUNS_FILE)

    # 2. Use safe status check from the existing db_config object
    safe_db_status = {item['setting']: item['configured'] for item in db_config.safe_status()}

    # 3. Handle status-dependent properties safely
    success_count = int((runs['status'] == 'success').sum()) if not runs.empty else 0
    partial_count = int((runs['status'] == 'partial').sum()) if not runs.empty else 0
    failed_count = int((runs['status'] == 'failed').sum()) if not runs.empty else 0

    # 4. Use 'configuration_id' instead of 'configuration_hash'
    unique_configs = 0
    if not runs.empty and 'configuration_id' in runs:
        unique_configs = int(runs.loc[runs['status'] == 'success', 'configuration_id'].nunique())

    # 5. Access optional objects safely using globals()
    labelled_len = int(len(globals()['labelled_dataset'])) if 'labelled_dataset' in globals() and isinstance(globals()['labelled_dataset'], pd.DataFrame) else 0
    best_model_exists = globals().get('best_model') is not None
    qdrant_ann_exists = globals().get('qdrant_ann_result') is not None
    beir_result_exists = globals().get('beir_result') is not None

    status = {
        "project_title": "ML-Driven NoSQL Pattern Selection for RAG Workloads",
        "updated_utc": datetime.now(timezone.utc).isoformat(),
        "runtime_report": runtime_report,
        "configured_secret_names": safe_db_status,
        "crud_run_rows": int(len(runs)),
        "unique_measured_configurations": unique_configs,
        "crud_success_runs": success_count,
        "crud_partial_runs": partial_count,
        "crud_failed_runs": failed_count,
        "labelled_workloads": labelled_len,
        "ml_trained": best_model_exists,
        "qdrant_ann_result_available": qdrant_ann_exists,
        "beir_result_available": beir_result_exists,
        "research_integrity_note": "Synthetic/demo rows must never be reported as database benchmark measurements.",
    }

    # 6. Ensure the target directory exists before saving
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    path = PROJECT_DIR / "PROJECT_STATUS.json"
    with open(path, "w") as f:
        json.dump(status, f, indent=2, default=str)
    return status

status_manifest = write_project_status()
print(json.dumps(status_manifest, indent=2))

{
  "project_title": "ML-Driven NoSQL Pattern Selection for RAG Workloads",
  "updated_utc": "2026-11-13T07:24:21.234465+00:00",
  "runtime_report": {
    "python": "3.13.16",
    "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
    "cpu_count": 2,
    "ram_total_gb": 12.67,
    "ram_available_gb": 10.71,
    "disk_free_gb": 112.33,
    "gpu_available": true,
    "gpu_name": "Tesla T4"
  },
  "configured_secret_names": {
    "redis_url": true,
    "mongodb_uri": true,
    "mode=astra": true,
    "neo4j_uri_and_auth": true,
    "qdrant_url": true
  },
  "crud_run_rows": 0,
  "unique_measured_configurations": 0,
  "crud_success_runs": 0,
  "crud_partial_runs": 0,
  "crud_failed_runs": 0,
  "labelled_workloads": 0,
  "ml_trained": false,
  "qdrant_ann_result_available": false,
  "beir_result_available": false,
  "research_integrity_note": "Synthetic/demo rows must never be reported as database benchmark measurements."
}


In [67]:
def create_artifact_manifest(root=PROJECT_DIR):
    rows=[]
    for path in Path(root).rglob("*"):
        if path.is_file() and path.name not in {"artifact_manifest.csv"}:
            stat=path.stat()
            rows.append({"relative_path":str(path.relative_to(root)),"size_bytes":stat.st_size,
                         "modified_utc":datetime.fromtimestamp(stat.st_mtime,timezone.utc).isoformat()})
    manifest=pd.DataFrame(rows).sort_values("relative_path") if rows else pd.DataFrame(columns=["relative_path","size_bytes","modified_utc"])
    manifest.to_csv(Path(root)/"artifact_manifest.csv",index=False)
    return manifest

artifact_manifest=create_artifact_manifest()
display(artifact_manifest.head(30))
print("Total files:",len(artifact_manifest))

,relative_path,size_bytes,modified_utc
1,PROJECT_STATUS.json,933,2026-11-13T07:24:21+00:00
4,datasets/raw/connection_checks.csv,311,2026-11-13T07:24:19+00:00
3,datasets/raw/smoke_test_synthetic_runs.csv,8452,2026-11-13T07:24:20+00:00
0,experiment_context.json,394,2026-11-13T07:24:02+00:00
2,experiments/logs/colab_experiment.log,0,2026-11-12T18:56:49+00:00


Total files: 5


In [68]:
import shutil

EXPORT_ZIP=Path("/content/NoSQL_RAG_Research_Artifacts.zip")
if EXPORT_ZIP.exists(): EXPORT_ZIP.unlink()
shutil.make_archive(str(EXPORT_ZIP.with_suffix("")), "zip", root_dir=PROJECT_DIR)
print("Created handoff archive:", EXPORT_ZIP)
print("Size MB:", round(EXPORT_ZIP.stat().st_size/(1024**2),2))

try:
    from google.colab import files
    files.download(str(EXPORT_ZIP))
except Exception:
    print("Not running in Colab. Archive is available at:", EXPORT_ZIP)

Created handoff archive: /content/NoSQL_RAG_Research_Artifacts.zip
Size MB: 0.0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 26. Final research validation checklist

Before reporting any result, verify the following. Mark each item only after evidence exists.

| Check | Required evidence | Status in a fresh runtime |
|---|---|---|
| Original research brief reviewed | Source document | To verify in project folder |
| Database endpoints configured | Secrets status + connection checks | Not yet configured |
| Redis CRUD adapter | Successful measured runs | Pending |
| MongoDB CRUD adapter | Successful measured runs | Pending |
| Cassandra CRUD adapter | Successful measured runs | Pending |
| Neo4j CRUD adapter | Successful measured runs | Pending |
| Qdrant ANN | Measured recall/latency output | Pending |
| 300+ unique workload/database configurations | Configuration-level records | Pending |
| Repeated trials | Run-level records per configuration | Pending |
| ML models | Saved models + held-out metrics | Pending |
| Latency regret | Held-out recommendation and measured latency | Pending |
| SHAP | Saved plots + feature names | Pending |
| BEIR retrieval | Dataset, qrels and retrieval metrics | Pending |
| Reproducibility | Environment/configuration/run metadata | Pending |
| End-to-end system | Executed tests and dashboard evidence | Not implemented in this notebook |

### Acceptance criteria

- No synthetic values in experimental result files.
- Raw data retained and immutable after capture.
- Each run has a configuration ID and run ID.
- Failed operations are visible in logs.
- Model features do not leak measured target metrics.
- Held-out evaluation is separated by workload profile.
- Regret is calculated from measured candidate latency for the same held-out workload.
- Vector retrieval quality is measured separately from CRUD latency.
- Report limitations match the actual experimental scope.

### Known implementation boundary

This notebook is the Colab experiment and analysis pipeline. A production-quality Streamlit dashboard, a fully automated five-engine Docker deployment, comprehensive graph benchmark suites, and vector indexes for multiple candidate engines are separate implementation tasks. Do not claim they are completed merely because this notebook covers the ML and retrieval workflow.

## 27. Next actions

1. Add the database connection secrets in Colab.
2. Run the connection-check cell and resolve every required failure.
3. Keep `RUN_PILOT=True` with the small pilot settings.
4. Inspect successful/failed runs and verify that the workload semantics are appropriate for each engine.
5. Confirm server versions, hosting region, TLS, indexes and configuration metadata.
6. Repeat pilot runs to check stability.
7. Only after pilot review, enable the full matrix in small resumable batches.
8. Train ML models only after the measured label dataset contains enough workloads and classes.
9. Run Qdrant ANN and BEIR retrieval as separate experiments.
10. Export the artifacts and use them as the evidence base for the final report.

**Do not fill in accuracy, regret, recall or latency claims in the research report until the corresponding experiments have actually run.**